In [ ]:
# VGG_PATIENT_STYLE_V1 — one Colab GPU cell; development-only candidate selection.
DATA_DIR = '/content/drive/MyDrive/T2_data'
SEED = 20261128
N_FOLDS = 5
MAX_EPOCHS = 8
SCHEDULE_EPOCHS = 11
RECIPES = ['patient_balanced', 'patient_mixstyle']
REFIT_ONLY_IF_CV_IMPROVES = True
HEAD_EPOCHS = 3
BATCH_SIZE = 8
KEEP_FOLD_WEIGHTS = False  # OOF predictions suffice for CV; save Drive space/time.

import importlib.util, subprocess, sys
missing = [p for m, p in {'numpy':'numpy', 'pandas':'pandas', 'PIL':'pillow',
                         'matplotlib':'matplotlib', 'tqdm':'tqdm'}.items()
           if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
import gc, hashlib, json, random, time, zipfile
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import importlib.metadata
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from google.colab import drive, files

IMAGE_SIZE = (336, 224)
MEAN, STD = [.485, .456, .406], [.229, .224, .225]
SMOOTHING = .03
OLD_VALID_GROUPS = {'G06','G07','G09','G14','G17','G20'}

def make_protocol(frame, seed):
    """Patient-balanced deterministic split using labels/counts, never model scores."""
    groups = frame.groupby('patient_group').agg(
        images=('label','size'), positives=('label','sum')).reset_index()
    assert len(groups) == 27 and len(frame) == 2948
    assert groups.positives.gt(0).all() and groups.images.gt(groups.positives).all()
    eligible = groups[~groups.patient_group.isin(OLD_VALID_GROUPS)].sort_values(['positives','patient_group'])
    assert len(eligible) == 21
    rng = np.random.default_rng(seed)
    test_groups = []
    # Six patients: two sampled from each of three positive-count strata.
    for stratum in np.array_split(eligible.patient_group.to_numpy(), 3):
        test_groups.extend(rng.choice(stratum, size=2, replace=False).tolist())
    development = groups[~groups.patient_group.isin(test_groups)].sort_values(
        ['positives','patient_group'], ascending=[False,True])
    permutation = rng.permutation(N_FOLDS)
    mapping = {}
    # Descending positive-count strata distributed in alternating directions.
    # This is a custom patient-balanced group split, not sklearn SGKF.
    for rank, group in enumerate(development.patient_group):
        offset = rank % N_FOLDS
        mapping[group] = int(permutation[offset] if (rank // N_FOLDS) % 2 == 0
                             else permutation[N_FOLDS - 1 - offset])
    result = frame.copy()
    result['split'] = np.where(result.patient_group.isin(test_groups), 'test', 'development')
    result['fold'] = result.patient_group.map(mapping).fillna(-1).astype(int)
    assert result.groupby('patient_group')['split'].nunique().eq(1).all()
    assert result.groupby('patient_group')['fold'].nunique().eq(1).all()
    assert not set(test_groups) & OLD_VALID_GROUPS
    for fold in range(N_FOLDS):
        tr = result[result.split.eq('development') & result.fold.ne(fold)]
        va = result[result.split.eq('development') & result.fold.eq(fold)]
        te = result[result.split.eq('test')]
        assert set(tr.label) == set(va.label) == set(te.label) == {0,1}
        assert not set(tr.patient_group) & set(va.patient_group)
        assert not set(te.patient_group) & (set(tr.patient_group) | set(va.patient_group))
    return result

def roc_arrays(y, p):
    y, p = np.asarray(y, int), np.asarray(p, float)
    assert set(np.unique(y)) == {0,1} and np.isfinite(p).all()
    order = np.argsort(-p, kind='stable')
    ys, ps = y[order], p[order]
    ends = np.r_[np.flatnonzero(ps[:-1] != ps[1:]), len(ps)-1]
    tp = np.r_[0, np.cumsum(ys)[ends]]
    fp = np.r_[0, np.cumsum(1-ys)[ends]]
    cuts = np.r_[np.nextafter(ps[0], np.inf),
                 (ps[ends[:-1]] + ps[ends[:-1]+1])/2, ps[-1]]
    return tp / y.sum(), fp / (y == 0).sum(), cuts

def scores(y, p, threshold=.5):
    y, p = np.asarray(y, int), np.asarray(p, float)
    tpr, fpr, _ = roc_arrays(y, p)
    pred = p >= threshold
    tn, fp = int((~pred[y==0]).sum()), int(pred[y==0].sum())
    fn, tp = int((~pred[y==1]).sum()), int(pred[y==1].sum())
    return dict(ba=float(.5*(tn/(tn+fp)+tp/(tp+fn))),
                auroc=float(np.trapezoid(tpr,fpr)),
                recall0=float(tn/(tn+fp)), recall1=float(tp/(tp+fn)),
                accuracy=float((pred==y).mean()), tn=tn, fp=fp, fn=fn, tp=tp)

def choose_development_threshold(y, p):
    tpr, fpr, cuts = roc_arrays(y,p)
    ba = .5*(tpr+1-fpr)
    options = np.flatnonzero(np.isclose(ba, ba.max(), atol=1e-12, rtol=0))
    return float(cuts[options[np.argmin(np.abs(cuts[options]-.5))]])

def seed_all(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def atomic_json(path, value):
    temporary = path.with_suffix(path.suffix+'.tmp')
    temporary.write_text(json.dumps(value, indent=2))
    temporary.replace(path)

class BalancedPatientBatches:
    def __init__(self,rows,seed):
        assert BATCH_SIZE == 8
        self.patients=np.array(sorted(rows.patient_group.unique()))
        assert len(self.patients)>=2
        self.pools={(g,c):np.flatnonzero((rows.patient_group.eq(g)&rows.label.eq(c)).to_numpy())
                    for g in self.patients for c in [0,1]}
        assert all(len(p)>0 for p in self.pools.values())
        self.steps=(len(rows)+BATCH_SIZE-1)//BATCH_SIZE
        self.seed=seed;self.epoch=0
    def __len__(self):return self.steps
    def __iter__(self):
        rng=np.random.default_rng(self.seed+self.epoch*1009)
        for _ in range(self.steps):
            patients=rng.choice(self.patients,2,replace=False)
            batch=[]
            for g in patients:
                for c in [0,1]:
                    pool=self.pools[(g,c)]
                    batch.extend(rng.choice(pool,2,replace=len(pool)<2).tolist())
            rng.shuffle(batch)
            yield batch

def cross_patient_partners(domains,rng=None):
    domains=np.asarray(domains)
    rng=np.random if rng is None else rng
    result=[]
    for i in range(len(domains)):
        available=np.flatnonzero(domains!=domains[i])
        if len(available)==0:return None
        result.append(int(rng.choice(available)))
    return np.asarray(result,dtype=np.int64)

class TrainingMixStyle:
    # Parameter-free feature augmentation; no inference layer or extra neural net.
    def __init__(self,model):self.model=model;self.patient_ids=None
    def __call__(self,module,inputs,output):
        if not self.model.training or self.patient_ids is None or torch.rand(()).item()>=.25:
            return output
        partners=cross_patient_partners(self.patient_ids.cpu().numpy())
        if partners is None:return output
        index=torch.tensor(partners,device=output.device,dtype=torch.long)
        x=output.float()
        mu=x.mean(dim=(2,3),keepdim=True).detach()
        sigma=(x.var(dim=(2,3),keepdim=True,unbiased=False)+1e-6).sqrt().detach()
        beta=torch.distributions.Beta(.3,.3).sample((len(x),1,1,1)).to(x.device)
        lam=1.-.5*(1.-beta)  # Retain at least 50% of the image's own statistics.
        mixed_mu=lam*mu+(1-lam)*mu[index]
        mixed_sigma=lam*sigma+(1-lam)*sigma[index]
        return (((x-mu)/sigma)*mixed_sigma+mixed_mu).to(output.dtype)

def run_groupcv():
    drive.mount('/content/drive')
    if not torch.cuda.is_available():
        raise RuntimeError('Colab GPU 런타임을 선택하세요.')
    assert N_FOLDS == 5 and MAX_EPOCHS >= 1 and HEAD_EPOCHS >= 1
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    seed_all(SEED)
    device = torch.device('cuda')
    source = Path(DATA_DIR)
    csvs = [source/'labels.csv'] if (source/'labels.csv').is_file() else list(source.rglob('labels.csv'))
    if len(csvs) != 1:
        raise ValueError('DATA_DIR 아래 labels.csv가 정확히 하나 있어야 합니다.')
    csv_path = csvs[0]
    frame = pd.read_csv(csv_path, dtype={'sample_id':str,'patient_group':str})
    columns = ['sample_id','path','label','patient_group']
    assert set(columns).issubset(frame.columns) and frame.sample_id.is_unique
    assert not frame[columns].isna().any().any() and set(frame.label) == {0,1}
    data_hash = hashlib.sha256(frame[columns].to_csv(index=False).encode()).hexdigest()
    protocol = make_protocol(frame, SEED)
    source_text=globals().get('_VGG_EXECUTED_SOURCE')
    if not isinstance(source_text,str) or 'VGG_PATIENT_STYLE_V1' not in source_text:
        try:source_text=get_ipython().history_manager.input_hist_raw[-1]
        except Exception:source_text='VGG_PATIENT_STYLE_V1: source is in the original notebook.'
    code_hash=hashlib.sha256(source_text.encode()).hexdigest()
    config = dict(architecture='torchvision.vgg16', initialization='IMAGENET1K_V1',
                  seed=SEED, n_folds=N_FOLDS, max_epochs=MAX_EPOCHS, head_epochs=HEAD_EPOCHS,
                  batch_size=BATCH_SIZE, image_size=list(IMAGE_SIZE), normalize_mean=MEAN,
                  normalize_std=STD, label_smoothing=SMOOTHING, data_hash=data_hash,code_hash=code_hash,
                  fine_lrs=[2e-6,1e-5,5e-6,5e-5],head_lrs=[1e-5,1e-4],weight_decay=1e-4,
                  scheduler='cosine with T_max=schedule_epochs',schedule_epochs=SCHEDULE_EPOCHS,
                  recipes=RECIPES, fine_loss='plain smoothed CE on patient-class-balanced minibatches',
                  batch_composition='2 patients x 2 classes x 2 images = 8',
                  mixstyle=dict(p=.25,alpha=.3,strength=.5,blocks=[4,9]),
                  test_evaluated=False,refit_only_if_cv_improves=REFIT_ONLY_IF_CV_IMPROVES,
                  augmentation=dict(horizontal_flip=.5,rotation=5,brightness=.1,contrast=.1),
                  split_method='patient positive-count strata; deterministic serpentine group folds',
                  epoch_selection='minimum mean fold balanced plain CE; earlier epoch breaks exact ties',
                  threshold_selection='one global threshold from selected-epoch development OOF only',
                  original_weights_used=False, ensemble=False,
                  caveat='Local holdout from previously used project data, not a new independent external Test.')
    config_hash = hashlib.sha256(json.dumps(config,sort_keys=True).encode()).hexdigest()
    out = source.parent/'task2_vgg_patientstyle'/f'v1_{config_hash[:12]}'
    out.mkdir(parents=True,exist_ok=True)
    report_zip = out/'vgg_patientstyle_report.zip'
    if (out/'experiment_complete.json').is_file():
        print('이미 완료한 동일 실험입니다. 저장된 개발용 CV 결과를 반환합니다.')
        print((out/'final_metrics.csv').read_text())
        print('Final weights:',out/'final'/'vgg.pt' if (out/'final'/'vgg.pt').is_file() else 'No refit: CV did not improve')
        if not zipfile.is_zipfile(report_zip):
            with zipfile.ZipFile(report_zip,'w',zipfile.ZIP_DEFLATED) as z:
                for p in out.rglob('*'):
                    if p.is_file() and p.suffix in {'.csv','.json','.txt','.png','.py'}:
                        z.write(p,str(p.relative_to(out)))
        files.download(str(report_zip))
        return
    if (out/'patient_protocol.csv').exists():
        saved = pd.read_csv(out/'patient_protocol.csv',dtype={'sample_id':str,'patient_group':str})
        pd.testing.assert_frame_equal(saved,protocol,check_dtype=False)
        assert json.loads((out/'config.json').read_text()) == config
    else:
        protocol.to_csv(out/'patient_protocol.csv',index=False)
        atomic_json(out/'config.json',config)
    atomic_json(out/'references.json',dict(
        MixStyle='https://arxiv.org/abs/2104.02008',
        authors_code='https://github.com/KaiyangZhou/mixstyle-release',
        adaptation='Training-only cross-patient statistics mixing; p=.25, alpha=.3, strength=.5 are project choices, not validated optima.',
        constraint='One final VGG16, no prediction or weight ensembling.'))
    development = protocol[protocol.split.eq('development')].reset_index(drop=True)
    test = protocol[protocol.split.eq('test')].reset_index(drop=True)
    summary = protocol.groupby(['split','fold']).agg(images=('sample_id','size'),
        patients=('patient_group','nunique'), positives=('label','sum'))
    summary.to_csv(out/'split_summary.csv')
    print(summary.to_string())
    print('Locked local Test:',sorted(test.patient_group.unique()))
    print('GPU:',torch.cuda.get_device_name(0),'Output:',out)
    print('완료된 fold는 재사용합니다. 중단된 fold는 처음부터 다시 시작합니다.')

    cache = {}
    patient_index={g:i for i,g in enumerate(sorted(development.patient_group.unique()))}
    def add_images(rows):
        def read_one(row):
            path = (csv_path.parent/str(row.path)).resolve()
            if not path.is_relative_to(csv_path.parent.resolve()) or not path.is_file():
                raise ValueError(f'잘못된 이미지 경로: {row.path}')
            with Image.open(path) as im:
                image = im.convert('RGB').resize((IMAGE_SIZE[1],IMAGE_SIZE[0]),Image.Resampling.BILINEAR)
            return row.sample_id,image
        missing_rows = rows[~rows.sample_id.isin(cache)]
        with ThreadPoolExecutor(max_workers=4) as executor:
            for sample_id,image in tqdm(executor.map(read_one,missing_rows.itertuples(index=False)),
                                        total=len(missing_rows),desc='Cache images'):
                cache[sample_id]=image
    # No Test image decoding/feature extraction until final evaluation.
    add_images(development)
    plain = transforms.Compose([transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
    augmented = transforms.Compose([transforms.RandomHorizontalFlip(.5),transforms.RandomRotation(5),
        transforms.ColorJitter(brightness=.1,contrast=.1),transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
    class Images(Dataset):
        def __init__(self,rows,training=False):
            self.rows=rows.reset_index(drop=True)
            self.transform=augmented if training else plain
        def __len__(self): return len(self.rows)
        def __getitem__(self,index):
            row=self.rows.iloc[index]
            return self.transform(cache[row.sample_id]),int(row.label),patient_index[row.patient_group]
    def loader(rows,training=False,seed=SEED,batch_size=BATCH_SIZE):
        return DataLoader(Images(rows,training),batch_size=batch_size,shuffle=training,
                          num_workers=2,pin_memory=True,
                          generator=torch.Generator().manual_seed(seed))
    @torch.inference_mode()
    def evaluate(model,rows):
        model.eval(); probabilities=[]; ce=[]
        for images,y,_ in loader(rows):
            images=images.to(device,non_blocking=True,memory_format=torch.channels_last)
            y=y.to(device,non_blocking=True)
            with torch.autocast('cuda',dtype=torch.float16): logits=model(images)
            probabilities.extend(logits.float().softmax(1)[:,1].cpu().numpy().tolist())
            ce.extend(F.cross_entropy(logits.float(),y,reduction='none').cpu().numpy().tolist())
        probabilities=np.asarray(probabilities); ce=np.asarray(ce)
        y=rows.label.to_numpy(int)
        return probabilities,dict(plain_ce=float(ce.mean()),
            balanced_ce=float(.5*(ce[y==0].mean()+ce[y==1].mean())))

    # One fresh ImageNet source for every fold and the final refit.
    reference=models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
    reference.classifier[-1]=nn.Linear(4096,2)
    initial_state={k:v.detach().cpu().clone() for k,v in reference.state_dict().items()}
    reference=reference.to(device=device,memory_format=torch.channels_last).eval()
    feature_chunks=[]
    with torch.inference_mode():
        for images,_,_ in tqdm(loader(development,batch_size=16),desc='Initial VGG feature cache'):
            images=images.to(device,non_blocking=True,memory_format=torch.channels_last)
            with torch.autocast('cuda',dtype=torch.float16):
                x=reference.avgpool(reference.features(images)).flatten(1)
            feature_chunks.append(x.cpu())
    pool_features=torch.cat(feature_chunks)
    del reference,feature_chunks,x,images
    gc.collect();torch.cuda.empty_cache()

    def class_weights(rows):
        counts=np.bincount(rows.label.to_numpy(int),minlength=2)
        return torch.tensor(len(rows)/(2*counts),device=device,dtype=torch.float32)
    def new_model(train_indices,seed):
        seed_all(seed)
        model=models.vgg16(weights=None)
        model.classifier[-1]=nn.Linear(4096,2)
        model.load_state_dict(initial_state)  # Never a previously fine-tuned checkpoint.
        model=model.to(device=device,memory_format=torch.channels_last)
        for parameter in model.parameters():parameter.requires_grad=False
        for parameter in model.classifier.parameters():parameter.requires_grad=True
        rows=development.iloc[train_indices].reset_index(drop=True)
        X=pool_features[train_indices].to(device).clone()
        y=torch.tensor(rows.label.to_numpy(int),device=device,dtype=torch.long)
        weights=class_weights(rows)
        optimizer=torch.optim.AdamW([
            {'params':model.classifier[:-1].parameters(),'lr':1e-5},
            {'params':model.classifier[-1].parameters(),'lr':1e-4}],weight_decay=1e-4)
        scaler=torch.amp.GradScaler('cuda')
        for epoch in range(HEAD_EPOCHS):
            model.eval();model.classifier.train()
            for batch in torch.randperm(len(rows),device=device).split(32):
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast('cuda',dtype=torch.float16):
                    losses=F.cross_entropy(model.classifier(X[batch]),y[batch],reduction='none',label_smoothing=SMOOTHING)
                    loss=(losses*weights[y[batch]]).mean()
                if not bool(torch.isfinite(loss).item()):raise RuntimeError('Non-finite head loss')
                scaler.scale(loss).backward();scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.classifier.parameters(),5.)
                scaler.step(optimizer);scaler.update()
        for parameter in model.parameters():parameter.requires_grad=True
        return model

    def fit(model,rows,epochs,seed,recipe,valid_rows=None,folder=None):
        seed_all(seed)
        assert recipe in RECIPES
        style=TrainingMixStyle(model) if recipe=='patient_mixstyle' else None
        handles=[model.features[i].register_forward_hook(style) for i in [4,9]] if style else []
        try:
            groups=[]
            for module,lr in [(model.features[:17],2e-6),(model.features[17:],1e-5),
                              (model.classifier[:-1],5e-6),(model.classifier[-1],5e-5)]:
                decay=[];no_decay=[]
                for name,p in module.named_parameters():
                    (no_decay if p.ndim==1 or name.endswith('bias') else decay).append(p)
                for params,wd in [(decay,1e-4),(no_decay,0.)]:
                    if params:groups.append(dict(params=params,lr=lr,weight_decay=wd))
            optimizer=torch.optim.AdamW(groups)
            # Fixed schedule: held-out fold labels cannot change this fold's learning rates.
            scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=SCHEDULE_EPOCHS,eta_min=1e-7)
            scaler=torch.amp.GradScaler('cuda')
            sampler=BalancedPatientBatches(rows.reset_index(drop=True),seed)
            train_loader=DataLoader(Images(rows,True),batch_sampler=sampler,num_workers=2,pin_memory=True,
                                    generator=torch.Generator().manual_seed(seed))
            history=[]; predictions=[]
            for epoch in range(1,epochs+1):
                sampler.epoch=epoch
                model.train();start=time.perf_counter();numerator=0.;denominator=0.
                for images,y,patient_ids in tqdm(train_loader,desc=f'{recipe} {epoch}/{epochs}',leave=False):
                    if style:style.patient_ids=patient_ids
                    images=images.to(device,non_blocking=True,memory_format=torch.channels_last)
                    y=y.to(device,non_blocking=True)
                    optimizer.zero_grad(set_to_none=True)
                    with torch.autocast('cuda',dtype=torch.float16):
                        losses=F.cross_entropy(model(images),y,reduction='none',label_smoothing=SMOOTHING)
                        loss=losses.mean()  # Sampler already balances patient/class; no double weighting.
                    if not bool(torch.isfinite(loss).item()):raise RuntimeError('Non-finite training loss')
                    scaler.scale(loss).backward();scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(),5.)
                    scaler.step(optimizer);scaler.update()
                    numerator+=float(losses.detach().sum().item());denominator+=len(y)
                scheduler.step()
                row=dict(epoch=epoch,train_smoothed_ce=numerator/denominator,
                         seconds=round(time.perf_counter()-start,2))
                if valid_rows is not None:
                    p,loss_metrics=evaluate(model,valid_rows)
                    predictions.append(p)
                    row.update(**loss_metrics,**scores(valid_rows.label.to_numpy(int),p))
                    print(f"Epoch {epoch:02d}: Valid BA@.5={row['ba']:.4f}, AUC={row['auroc']:.4f}, balanced CE={row['balanced_ce']:.4f}",flush=True)
                else:print(f"Final refit {epoch:02d}/{epochs}",flush=True)
                history.append(row)
                if folder is not None:pd.DataFrame(history).to_csv(folder/'history.csv',index=False)
            return history,predictions
        finally:
            for handle in handles:handle.remove()

    oof_by_recipe={r:np.full((MAX_EPOCHS,len(development)),np.nan) for r in RECIPES}
    history_by_recipe={r:[] for r in RECIPES}
    for fold in range(N_FOLDS):
        train_indices=np.flatnonzero(development.fold.ne(fold).to_numpy())
        valid_indices=np.flatnonzero(development.fold.eq(fold).to_numpy())
        tr=development.iloc[train_indices].reset_index(drop=True)
        va=development.iloc[valid_indices].reset_index(drop=True)
        assert not set(tr.patient_group)&set(va.patient_group)
        assert not set(test.patient_group)&(set(tr.patient_group)|set(va.patient_group))
        pending=[r for r in RECIPES if not (out/r/f'fold_{fold}'/'complete.json').is_file()]
        ready_state=None
        if pending:
            print(f'\nFold {fold}: common FC preparation for {pending}')
            ready_model=new_model(train_indices,SEED+fold)
            ready_state={k:v.detach().cpu().clone() for k,v in ready_model.state_dict().items()}
            del ready_model;gc.collect();torch.cuda.empty_cache()
        for recipe in RECIPES:
            folder=out/recipe/f'fold_{fold}';folder.mkdir(parents=True,exist_ok=True)
            if (folder/'complete.json').is_file():
                assert json.loads((folder/'complete.json').read_text())['config_hash']==config_hash
                with np.load(folder/'validation_by_epoch.npz',allow_pickle=False) as saved:
                    assert np.array_equal(saved['sample_id'],va.sample_id.to_numpy(str))
                    assert np.array_equal(saved['label'],va.label.to_numpy(int))
                    probabilities=saved['probabilities'].copy()
                history=pd.read_csv(folder/'history.csv').to_dict('records')
                assert probabilities.shape==(MAX_EPOCHS,len(va)) and len(history)==MAX_EPOCHS
                print(f'{recipe} fold {fold}: completed results reused')
            else:
                model=models.vgg16(weights=None);model.classifier[-1]=nn.Linear(4096,2)
                model.load_state_dict(ready_state)
                model=model.to(device=device,memory_format=torch.channels_last)
                history,probabilities=fit(model,tr,MAX_EPOCHS,SEED+fold,recipe,va,folder)
                probabilities=np.asarray(probabilities)
                np.savez_compressed(folder/'validation_by_epoch.npz',sample_id=va.sample_id.to_numpy(str),
                                    label=va.label.to_numpy(int),probabilities=probabilities)
                if KEEP_FOLD_WEIGHTS:
                    torch.save(dict(model_state={k:v.detach().cpu() for k,v in model.state_dict().items()},config=config),folder/'diagnostic_fold.pt')
                atomic_json(folder/'complete.json',dict(config_hash=config_hash))
                del model;gc.collect();torch.cuda.empty_cache()
            oof_by_recipe[recipe][:,valid_indices]=probabilities
            history_by_recipe[recipe].extend(dict(fold=fold,**row) for row in history)
        del ready_state
    candidates=[];selected_data={}
    for recipe in RECIPES:
        folder=out/recipe
        oof=oof_by_recipe[recipe];assert np.isfinite(oof).all()
        histories=pd.DataFrame(history_by_recipe[recipe]);histories.to_csv(folder/'cv_history.csv',index=False)
        curve=histories.groupby('epoch').agg(mean_balanced_ce=('balanced_ce','mean'),
            std_balanced_ce=('balanced_ce','std'),mean_ba_05=('ba','mean'),mean_auroc=('auroc','mean'))
        curve.to_csv(folder/'cv_epoch_summary.csv')
        selected_epoch=int(curve.mean_balanced_ce.idxmin())
        selected_oof=oof[selected_epoch-1]
        threshold=choose_development_threshold(development.label.to_numpy(int),selected_oof)
        oof_frame=development[['sample_id','patient_group','label','fold']].copy()
        oof_frame['probability']=selected_oof;oof_frame['prediction']=(selected_oof>=threshold).astype(int)
        oof_frame.to_csv(folder/'development_oof.csv',index=False)
        fold_rows=[]
        for f,part in oof_frame.groupby('fold'):
            fold_rows.append(dict(fold=int(f),**scores(part.label.to_numpy(int),part.probability.to_numpy(float),threshold)))
        ft=pd.DataFrame(fold_rows);ft.to_csv(folder/'cv_fold_metrics.csv',index=False)
        patient_rows=[]
        for g,part in oof_frame.groupby('patient_group'):
            patient_rows.append(dict(patient_group=g,**scores(part.label.to_numpy(int),part.probability.to_numpy(float),threshold)))
        pt=pd.DataFrame(patient_rows);pt.to_csv(folder/'patient_metrics.csv',index=False)
        row=dict(recipe=recipe,selected_epoch=selected_epoch,threshold=threshold,
                 mean_fold_ba=float(ft.ba.mean()),std_fold_ba=float(ft.ba.std(ddof=1)),
                 worst_fold_ba=float(ft.ba.min()),mean_fold_auroc=float(ft.auroc.mean()),
                 pooled_oof_ba=scores(development.label.to_numpy(int),selected_oof,threshold)['ba'],
                 pooled_oof_auroc=scores(development.label.to_numpy(int),selected_oof,threshold)['auroc'])
        atomic_json(folder/'cv_summary.json',row);candidates.append(row)
        selected_data[recipe]=(histories,curve,oof_frame,selected_oof)
    comparison=pd.DataFrame(candidates);comparison.to_csv(out/'candidate_comparison.csv',index=False)
    winner=max(candidates,key=lambda r:(r['mean_fold_ba'],r['worst_fold_ba'],r['mean_fold_auroc']))
    recipe=winner['recipe'];selected_epoch=winner['selected_epoch'];threshold=winner['threshold']
    histories,curve,oof_frame,selected_oof=selected_data[recipe]
    selection=dict(selected_recipe=recipe,selected_epoch=selected_epoch,threshold=threshold,
        epoch_source='development CV mean balanced plain CE',threshold_source='development OOF',
        candidate_source='mean fold BA, then worst fold BA, then mean AUROC',
        test_used_for_selection=False,test_evaluated=False,
        previous_mean_fold_ba=.8500322787592525,
        exceeds_previous_cv=bool(winner['mean_fold_ba']>.8500322787592525))
    atomic_json(out/'selection.json',selection)
    print('\n',comparison.to_string(index=False))
    print('Test was not used or re-evaluated.')
    do_refit=selection['exceeds_previous_cv'] or not REFIT_ONLY_IF_CV_IMPROVES
    final_dir=out/'final';final_dir.mkdir(exist_ok=True)
    final_file=final_dir/'vgg.pt'
    results=[dict(split='selected_development_oof',**scores(development.label.to_numpy(int),selected_oof,threshold))]
    if do_refit:
        if final_file.is_file():
            checkpoint=torch.load(final_file,map_location='cpu',weights_only=True)
            assert checkpoint['config_hash']==config_hash and checkpoint['selection']==selection
            model=models.vgg16(weights=None);model.classifier[-1]=nn.Linear(4096,2)
            model.load_state_dict(checkpoint['model_state']);model=model.to(device=device,memory_format=torch.channels_last)
            del checkpoint
        else:
            model=new_model(np.arange(len(development)),SEED+100)
            fit(model,development,selected_epoch,SEED+100,recipe,folder=final_dir)
            final_config=dict(config,label_hash=data_hash,selected_recipe=recipe,
                train_groups=sorted(development.patient_group.unique().tolist()),
                reserved_test_groups=sorted(test.patient_group.unique().tolist()),
                class_names=['absent','present'],positive_class=1,
                inference='standard VGG16; MixStyle removed, no patient metadata needed')
            temporary=final_dir/'vgg.pt.tmp'
            torch.save(dict(model_state={k:v.detach().cpu() for k,v in model.state_dict().items()},
                config=final_config,threshold=threshold,selection=selection,config_hash=config_hash),temporary)
            temporary.replace(final_file)
        train_p,train_loss=evaluate(model,development)
        results.append(dict(split='development_train_final_model',**scores(development.label.to_numpy(int),train_p,threshold),**train_loss))
        del model;gc.collect();torch.cuda.empty_cache()
    else:
        print('기존 CV 평균 BA 85.00%를 넘지 못해 최종 재학습은 생략합니다. 기존 가중치를 보존하세요.')
    pd.DataFrame(results).to_csv(out/'final_metrics.csv',index=False)
    (out/'requirements_actual.txt').write_text('\n'.join(
        f'{p}=={importlib.metadata.version(p)}' for p in ['torch','torchvision','numpy','pandas','pillow','matplotlib','tqdm'])+'\n')
    fig,axes=plt.subplots(1,2,figsize=(11,4))
    for fold,part in histories.groupby('fold'):
        axes[0].plot(part.epoch,part.balanced_ce,alpha=.45,label=f'fold {fold}')
        axes[1].plot(part.epoch,part.auroc,alpha=.45,label=f'fold {fold}')
    axes[0].plot(curve.index,curve.mean_balanced_ce,color='black',lw=2,label='fold mean')
    for ax in axes:ax.axvline(selected_epoch,color='gray',ls='--');ax.set_xlabel('Epoch');ax.legend(fontsize=8)
    axes[0].set_ylabel('Validation balanced plain CE');axes[1].set_ylabel('Validation AUROC')
    fig.tight_layout();fig.savefig(out/'cv_curves.png',dpi=140);plt.show();plt.close(fig)
    inference_source="import numpy as np\nimport pandas as pd\nimport torch\nfrom pathlib import Path\nfrom PIL import Image\nfrom torchvision import models, transforms\n\ndef predict_csv(weights_path, csv_path, image_root, output_csv='predictions.csv'):\n    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\n    checkpoint = torch.load(weights_path, map_location='cpu', weights_only=True)\n    config = checkpoint['config']\n    architecture = config['architecture'].split('.')[-1]\n    if architecture not in {'vgg11', 'vgg16'}:\n        raise ValueError('Unsupported VGG architecture in checkpoint.')\n    model = getattr(models, architecture)(weights=None)\n    model.classifier[-1] = torch.nn.Linear(4096, 2)\n    model.load_state_dict(checkpoint['model_state'])\n    model = model.to(device=device, memory_format=torch.channels_last).eval()\n    threshold = float(checkpoint['threshold'])\n    frame = pd.read_csv(csv_path, dtype={'sample_id': str})\n    transform = transforms.Compose([transforms.ToTensor(),\n        transforms.Normalize(config['normalize_mean'], config['normalize_std'])])\n    height, width = config['image_size']\n    probabilities = []\n    with torch.inference_mode():\n        for start in range(0, len(frame), 8):\n            images = []\n            for p in frame.iloc[start:start+8].path:\n                with Image.open(Path(image_root) / str(p)) as im:\n                    im = im.convert('RGB').resize((width, height), Image.Resampling.BILINEAR)\n                    images.append(transform(im))\n            batch = torch.stack(images).to(device=device, memory_format=torch.channels_last)\n            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == 'cuda'):\n                logits = model(batch)\n            probabilities.extend(logits.float().softmax(1)[:, 1].cpu().numpy().tolist())\n    result = frame[['sample_id']].copy()\n    result['probability'] = probabilities\n    result['prediction'] = (np.asarray(probabilities) >= threshold).astype(int)\n    result.to_csv(output_csv, index=False)\n    return result\n"
    (out/'inference.py').write_text(inference_source)
    (out/'executed_cell.py').write_text(source_text)
    atomic_json(out/'experiment_complete.json',dict(config_hash=config_hash,selection=selection,
        final_refit=do_refit,test_evaluated=False,
        note='All candidate selection uses development CV. Existing local Test was not re-evaluated.'))
    with zipfile.ZipFile(report_zip,'w',zipfile.ZIP_DEFLATED) as z:
        for p in out.rglob('*'):
            if p.is_file() and p.suffix in {'.csv','.json','.txt','.png','.py'}:
                z.write(p,str(p.relative_to(out)))
    print('\n',pd.DataFrame(results).to_string(index=False))
    print('최종 단일 모델:',final_file if do_refit else 'not refit; no CV improvement')
    print('선택 epoch:',selected_epoch,'공통 임계값:',threshold)
    print('이 셀은 Test 영상을 읽거나 Test를 재평가하지 않습니다. 최종 추론은 VGG 하나입니다.')
    print('이 로컬 Test는 이미 개발에 사용된 과제 데이터의 재분할입니다. 교수님의 private Test와 다릅니다.')
    del initial_state,pool_features,cache
    gc.collect();torch.cuda.empty_cache()
    files.download(str(report_zip))

run_groupcv()
